# ✅ Model Approval Gateway

## Purpose
Checks for human approval before promoting Challenger to Champion

## Workflow
1. Load Challenger model metadata
2. Check for `approval=approved` tag on model version
3. Fail if approval tag is not present
4. Block downstream deployment until approval is granted

## Notes
- Part of 3-step deployment workflow (Evaluation → **Approval** → Deployment)
- Follows Iris MLOps pattern for human oversight
- Approval is granted by setting tag: `approval=approved` on model version in Unity Catalog UI
- Safety gate for production deployment

In [ ]:
# 📦 Install required packages
%pip install mlflow

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import SparkSession

# Initialize
spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

print("✅ Libraries loaded for approval check")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")
model_name = spark.conf.get("model_name", "water_quality_model")

# Construct full model name
FULL_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"

print(f"🎯 Approval Check Configuration:")
print(f"   🤖 Model: {FULL_MODEL_NAME}")
print(f"   🏷️ Checking: Challenger model approval status")

In [ ]:
# 🔍 Get Challenger Model Version
print("🔍 Loading Challenger model information...")

try:
    # Get the Challenger model version
    model_version = client.get_model_version_by_alias(FULL_MODEL_NAME, "Challenger")
    
    print(f"✅ Found Challenger model:")
    print(f"   📦 Version: {model_version.version}")
    print(f"   🏷️ Alias: Challenger")
    print(f"   📅 Created: {model_version.creation_timestamp}")
    
except Exception as e:
    print(f"❌ Error: Could not find Challenger model")
    print(f"🔍 Details: {e}")
    print("💡 Make sure training job completed successfully")
    raise e

In [ ]:
# 🔍 Check Model Tags and Status
print("🔍 Checking model tags and evaluation status...")

# Get all tags for this model version
try:
    model_version_details = client.get_model_version(FULL_MODEL_NAME, model_version.version)
    tags = model_version_details.tags or {}
    
    print(f"📋 Current model tags:")
    for key, value in tags.items():
        print(f"   {key}: {value}")
    
    # Check evaluation status first
    eval_status = tags.get("evaluation_status", "not_evaluated")
    print(f"\n🔬 Evaluation Status: {eval_status}")
    
    if eval_status != "passed":
        print(f"❌ Model evaluation has not passed!")
        print(f"💡 Current status: {eval_status}")
        print(f"🔄 Run model evaluation first")
        raise Exception(f"Model evaluation required. Current status: {eval_status}")
    
    print(f"✅ Model evaluation: PASSED")
    
except Exception as e:
    print(f"❌ Error checking model version: {e}")
    raise e

In [ ]:
# ✅ Check Human Approval
print("✅ Checking for human approval...")

# Check for approval tag
approval_status = tags.get("approval", "not_set")
approval_by = tags.get("approved_by", "unknown")
approval_date = tags.get("approval_date", "unknown")

print(f"🔍 Approval Status: {approval_status}")

if approval_status == "approved":
    print(f"✅ MODEL APPROVED FOR PRODUCTION!")
    print(f"👤 Approved by: {approval_by}")
    print(f"📅 Approval date: {approval_date}")
    
    # Add approval confirmation tag
    client.set_model_version_tag(
        name=FULL_MODEL_NAME,
        version=model_version.version,
        key="deployment_ready",
        value="true"
    )
    
    print(f"🎯 Model is ready for deployment to Champion")
    
elif approval_status in ["rejected", "denied"]:
    print(f"❌ MODEL REJECTED FOR PRODUCTION")
    print(f"👤 Rejected by: {approval_by}")
    print(f"📅 Rejection date: {approval_date}")
    print(f"🔄 Model needs retraining or fixes")
    raise Exception(f"Model deployment rejected by: {approval_by}")
    
else:
    print(f"⏳ MODEL AWAITING HUMAN APPROVAL")
    print(f"📋 Current approval status: {approval_status}")
    print(f"")
    print(f"🎯 TO APPROVE THIS MODEL:")
    print(f"   1. Go to Unity Catalog → Models → {FULL_MODEL_NAME}")
    print(f"   2. Select Version {model_version.version}")
    print(f"   3. Add tags:")
    print(f"      - approval: approved")
    print(f"      - approved_by: your_name")
    print(f"      - approval_date: {pd.Timestamp.now().strftime('%Y-%m-%d')}")
    print(f"   4. Re-run this approval job")
    print(f"")
    print(f"🔄 TO REJECT THIS MODEL:")
    print(f"   1. Add tag: approval: rejected") 
    print(f"   2. Add tag: rejected_by: your_name")
    print(f"")
    
    # Block deployment
    raise Exception(f"Model approval required. Current status: {approval_status}. Set approval=approved tag to proceed.")

In [ ]:
# 📊 Approval Summary
print("📊 APPROVAL SUMMARY:")
print("=" * 40)
print(f"🤖 Model: {FULL_MODEL_NAME}")
print(f"📦 Version: {model_version.version}")
print(f"🔬 Evaluation: ✅ PASSED")
print(f"✅ Approval: ✅ APPROVED")
print(f"👤 Approved by: {approval_by}")
print(f"📅 Date: {approval_date}")
print(f"")
print(f"🎯 NEXT: Run deployment job to promote to Champion")

# Import pandas for date formatting
import pandas as pd

# Log approval success
print(f"✅ APPROVAL CHECK COMPLETED SUCCESSFULLY!")
print(f"🚀 Model ready for Champion promotion and endpoint deployment")